# Industrial Production Time-Series Forecasting

Portfolio notebook for forecasting the Spanish **Industrial Production Index (IPI)** with classical univariate time-series methods.

This public version focuses on the complete analytical workflow:

- monthly time-series construction;
- trend and seasonal decomposition;
- Augmented Dickey–Fuller testing;
- chronological 12-month holdout validation;
- Exponential Smoothing model selection;
- Seasonal ARIMA selection with `auto_arima`;
- MAE, RMSE and MAPE comparison;
- Ljung–Box and residual-autocorrelation diagnostics.

The source workbook is not redistributed in the public repository.

## 1. Imports and data loading

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

DATA_PATH = Path("../data/IPI_Esp.xlsx")
data = pd.read_excel(DATA_PATH)

data.head()

In [ ]:
dates = pd.PeriodIndex(
    data["Date"].astype(str).str.replace("M", "-", regex=False),
    freq="M",
).to_timestamp()

ipi = pd.Series(
    data["IPI Nacional"].astype(float).to_numpy(),
    index=dates,
    name="IPI_Nacional",
)

print(f"Observations: {len(ipi):,}")
print(f"Start: {ipi.index.min().date()}")
print(f"End: {ipi.index.max().date()}")

**Completed-run summary**

- Observations: **539**
- Start: **1975-01**
- End: **2019-11**

## 2. Time-series structure

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(ipi.index, ipi.values)
plt.title("Spanish Industrial Production Index")
plt.xlabel("Date")
plt.ylabel("IPI")
plt.tight_layout()
plt.show()

![Historical IPI series](../images/series_history.png)

The historical series shows long-term movement together with a strong recurring annual seasonal pattern.

In [ ]:
decomposition = seasonal_decompose(
    ipi,
    model="additive",
    period=12,
    extrapolate_trend="freq",
)

plt.figure(figsize=(11, 4))
plt.plot(decomposition.trend.index, decomposition.trend.values)
plt.title("Estimated Trend Component")
plt.xlabel("Date")
plt.ylabel("Trend")
plt.tight_layout()
plt.show()

![Trend component](../images/trend_component.png)

In [ ]:
seasonal_view = decomposition.seasonal.iloc[:24]

plt.figure(figsize=(11, 4))
plt.plot(seasonal_view.index, seasonal_view.values, marker="o")
plt.title("Seasonal Component — Two-Year View")
plt.xlabel("Date")
plt.ylabel("Seasonal effect")
plt.tight_layout()
plt.show()

![Seasonal component](../images/seasonal_component.png)

## 3. Stationarity

In [ ]:
def adf_summary(series, label):
    result = adfuller(series.dropna(), autolag="AIC")
    return {
        "series_version": label,
        "adf_statistic": result[0],
        "p_value": result[1],
    }

adf_results = pd.DataFrame([
    adf_summary(ipi, "Level"),
    adf_summary(ipi.diff(), "First difference"),
    adf_summary(ipi.diff(12), "Seasonal difference (12)"),
])

adf_results

**Completed-run result**

The ADF p-value for the level series is approximately **0.3898**, so the unit-root null is not rejected at conventional significance levels.

Differencing produces much stronger evidence of stationarity.

## 4. Chronological validation design

In [ ]:
HORIZON = 12

train = ipi.iloc[:-HORIZON]
test = ipi.iloc[-HORIZON:]

print(f"TRAIN observations: {len(train)}")
print(f"TEST observations: {len(test)}")
print(f"TEST start: {test.index.min().date()}")
print(f"TEST end: {test.index.max().date()}")

The final **12 months** are reserved as TEST.

This preserves chronology and gives both forecasting approaches the **same one-year seasonal evaluation window**.

## 5. Exponential Smoothing

In [ ]:
ets_candidates = []

for trend in [None, "add"]:
    for seasonal in ["add", "mul"]:
        damped_options = [False, True] if trend is not None else [False]

        for damped in damped_options:
            fit = ExponentialSmoothing(
                train,
                trend=trend,
                damped_trend=damped,
                seasonal=seasonal,
                seasonal_periods=12,
                initialization_method="estimated",
            ).fit(
                optimized=True,
                use_brute=True,
            )

            ets_candidates.append({
                "trend": trend,
                "seasonal": seasonal,
                "damped": damped,
                "AIC": fit.aic,
                "fit": fit,
            })

ets_ranking = (
    pd.DataFrame(ets_candidates)
    .drop(columns="fit")
    .sort_values("AIC")
)

ets_ranking

In [ ]:
best_ets = min(ets_candidates, key=lambda row: row["AIC"])
ets_fit = best_ets["fit"]

ets_forecast = pd.Series(
    ets_fit.forecast(HORIZON),
    index=test.index,
    name="ETS",
)

print(best_ets["trend"], best_ets["seasonal"], best_ets["damped"])

**Selected ETS specification**

- additive trend;
- damped trend;
- multiplicative seasonality;
- seasonal period = 12.

## 6. Seasonal ARIMA

In [ ]:
sarima_search = auto_arima(
    train,
    seasonal=True,
    m=12,
    start_p=0,
    start_q=0,
    max_p=3,
    max_q=3,
    start_P=0,
    start_Q=0,
    max_P=2,
    max_Q=2,
    d=None,
    D=None,
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
    information_criterion="aic",
    with_intercept="auto",
)

print("Order:", sarima_search.order)
print("Seasonal order:", sarima_search.seasonal_order)
print("AIC:", sarima_search.aic())

**Selected SARIMA Specification**

The model-selection process identified:

**SARIMA(2,1,1) × (2,0,2,12)**

Model selection is performed exclusively on the TRAIN partition, preserving the TEST period for final out-of-sample evaluation.

In [ ]:
sarima_forecast = pd.Series(
    sarima_search.predict(n_periods=HORIZON),
    index=test.index,
    name="SARIMA",
)

## 7. Out-of-sample comparison

In [ ]:
def forecast_metrics(actual, predicted):
    actual = pd.Series(actual, dtype=float)
    predicted = pd.Series(predicted, index=actual.index, dtype=float)

    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "MAPE (%)": (
            np.mean(np.abs((actual - predicted) / actual)) * 100
        ),
    }

metrics = pd.DataFrame({
    "Exponential Smoothing": forecast_metrics(test, ets_forecast),
    "SARIMA": forecast_metrics(test, sarima_forecast),
}).T

metrics

**Completed-run TEST metrics**

| Model | MAE | RMSE | MAPE |
|---|---:|---:|---:|
| Exponential Smoothing | 2.242 | 2.840 | **2.10%** |
| SARIMA | 2.204 | 2.719 | **2.06%** |

The two models perform similarly, with SARIMA showing a modest advantage in RMSE and MAPE.

In [ ]:
history_window = train.iloc[-48:]

plt.figure(figsize=(11, 5))
plt.plot(history_window.index, history_window.values, label="Train")
plt.plot(test.index, test.values, marker="o", label="Test actual")
plt.plot(test.index, ets_forecast.values, marker="o", label="ETS forecast")
plt.plot(test.index, sarima_forecast.values, marker="o", label="SARIMA forecast")
plt.title("12-Month Holdout Forecast Comparison")
plt.xlabel("Date")
plt.ylabel("IPI")
plt.legend()
plt.tight_layout()
plt.show()

![Forecast comparison](../images/forecast_comparison.png)

## 8. Residual diagnostics

In [ ]:
ets_residuals = pd.Series(ets_fit.resid).dropna()
sarima_residuals = pd.Series(sarima_search.resid()).dropna()

diagnostics = []

for model_name, residuals in [
    ("Exponential Smoothing", ets_residuals),
    ("SARIMA", sarima_residuals),
]:
    lb = acorr_ljungbox(
        residuals,
        lags=[12, 24],
        return_df=True,
    )

    for lag in [12, 24]:
        diagnostics.append({
            "Model": model_name,
            "Lag": lag,
            "Ljung-Box statistic": lb.loc[lag, "lb_stat"],
            "p-value": lb.loc[lag, "lb_pvalue"],
        })

pd.DataFrame(diagnostics)

In [ ]:
plot_acf(sarima_residuals, lags=36)
plt.title("SARIMA Residual Autocorrelation")
plt.tight_layout()
plt.show()

![SARIMA residual ACF](../images/sarima_residual_acf.png)

Residual autocorrelation remains detectable at seasonal horizons.

This is explicitly retained as a limitation: strong forecast metrics do not imply that the residual process is perfectly white noise.

## 9. Conclusions

- The level IPI series is not supported as stationary by the ADF test.
- The monthly series shows strong annual seasonality.
- Chronological validation is required for this forecasting problem.
- A 12-month holdout gives one complete seasonal cycle for evaluation.
- ETS and SARIMA achieve similar out-of-sample performance.
- SARIMA has a modest advantage on RMSE and MAPE on this holdout.
- Residual diagnostics still show remaining temporal dependence.

## Public portfolio boundary

The source workbook is not redistributed.

The notebook is designed to be readable directly on GitHub. Code cells show the full forecasting workflow, while the repository images and completed-run summaries make the analytical results visible without requiring execution.

To reproduce the analysis, place the source workbook at:

`data/IPI_Esp.xlsx`